# FDB fraudecom champion - Exp 6 XGBoost

Self-contained notebook to train + evaluate the chronological-holdout champion.


In [ ]:
!pip install xgboost==3.2.0 pandas numpy scikit-learn -q

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score
import xgboost as xgb
# Replace with your local copy
df = pd.read_csv('features_velocity.csv')
n=len(df); n_test=int(round(n*0.2)); n_val=int(round(n*0.1)); n_train=n-n_val-n_test
X = df.drop(columns=['class']).to_numpy(float); y = df['class'].to_numpy(int)
mu=X[:n_train].mean(0); sd=X[:n_train].std(0)+1e-8; Xs=(X-mu)/sd
Xtr,ytr = Xs[:n_train], y[:n_train]
Xva,yva = Xs[n_train:n_train+n_val], y[n_train:n_train+n_val]
Xte,yte = Xs[n_train+n_val:], y[n_train+n_val:]
clf = xgb.XGBClassifier(n_estimators=600, max_depth=6, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0, min_child_weight=5,
    random_state=0, tree_method='hist', n_jobs=4, early_stopping_rounds=40)
clf.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
p = clf.predict_proba(Xte)[:,1]
print('Test AUC:', roc_auc_score(yte, p))  # expect ~0.5414